# 02 — MC Dropout Inference on Test Set

Runs the trained EEGNet with Monte Carlo dropout (N stochastic passes) over the held-out test subjects and saves the per-epoch confidences and predictions for the calibration analysis in `03_calibration.ipynb`.

Requires the cloud/local training artifacts in `checkpoints/`: `eegnet.pt`, `X_test.npy`, `test_labels.npy`.

In [1]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from src.models.mc_dropout import MCDropoutDecoder, STATES

CHECKPOINT = "checkpoints/eegnet.pt"
N_PASSES = 50

In [2]:
probs_baseline = np.load("checkpoints/test_probs_baseline.npy")  # (n_test, 3)
y_test = np.load("checkpoints/test_labels.npy")                  # (n_test,)
print(f"Test set: {len(y_test)} epochs, class dist: {np.bincount(y_test)}")

Test set: 1695 epochs, class dist: [824 439 432]


In [3]:
decoder = MCDropoutDecoder(CHECKPOINT, n_passes=N_PASSES, device="cpu")
X_test = np.load("checkpoints/X_test.npy")

mc_states, mc_confidences = [], []
for epoch in X_test:
    state, conf = decoder.predict(epoch.astype(np.float32))
    mc_states.append(state)
    mc_confidences.append(conf)

mc_preds = np.array([STATES.index(s) for s in mc_states])
mc_confs = np.array(mc_confidences)
mc_acc = (mc_preds == y_test).mean()
print(f"MC dropout accuracy: {mc_acc:.3f}")

/Users/samvrithbandi/Desktop/personal-proj/neuroMCP/.venv/lib/python3.14/site-packages/torch/nn/modules/conv.py:560: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/native/Convolution.cpp:1091.)
  return F.conv2d(


MC dropout accuracy: 0.577


In [4]:
# Persist for the calibration notebook (03_calibration.ipynb)
np.save("checkpoints/mc_confidences.npy", mc_confs)
np.save("checkpoints/mc_preds.npy", mc_preds)
print("Saved checkpoints/mc_confidences.npy and checkpoints/mc_preds.npy")

Saved checkpoints/mc_confidences.npy and checkpoints/mc_preds.npy
